In [5]:
# import libraries
import cobra
from cobra.flux_analysis import flux_variability_analysis

# import model
model = cobra.io.read_sbml_model("../models/model_3.xml")

In [6]:
# Identify reactions that loop infinitely even when starved
## All exchanges closed, no maintenance, objective = 0
## Done inside a context manager so these constraints are NOT saved
## this step is purely diagnostic.

print("\n--- Sealing model & relaxing constraints to find internal loops ---")
internal_loops = None
with model:
    for rxn in model.exchanges:
        rxn.lower_bound = 0
    print(f"Closed {len(model.exchanges)} exchange reactions. Model is now starved.")

    for rxn in model.reactions:
        if "ATPM" in rxn.id.upper() or "MAINT" in rxn.id.upper():
            rxn.lower_bound = 0
            print(f"Relaxed maintenance constraint for: {rxn.id}")

    model.objective = model.problem.Objective(0)

    try:
        fva_closed = flux_variability_analysis(model, fraction_of_optimum=0)
        threshold = 999
        unbounded_mask = (fva_closed['maximum'] >= threshold) | (fva_closed['minimum'] <= -threshold)
        internal_loops = fva_closed[unbounded_mask]
        print(f"Found {len(internal_loops)} reactions that loop infinitely even without food.")
        if len(internal_loops) > 0:
            print("\nTop 15 looping reactions to investigate:")
            print(internal_loops)
    except Exception as e:
        print(f"FVA failed: {e}")
        print("There is still a hardcoded forced-flux reaction preventing the model from resting at 0.")



--- Sealing model & relaxing constraints to find internal loops ---
Closed 252 exchange reactions. Model is now starved.
Relaxed maintenance constraint for: ATPM
Found 27 reactions that loop infinitely even without food.

Top 15 looping reactions to investigate:
                 minimum  maximum
5DGLCNR     0.000000e+00   1000.0
5DKGR       0.000000e+00   1000.0
ACTD        0.000000e+00   1000.0
ACTD_1      0.000000e+00   1000.0
ACTDa       0.000000e+00   1000.0
ALCD2ir    -1.000000e+01   1000.0
ALCD2x     -1.000000e+01   1000.0
ALCD2y     -1.000000e+01   1000.0
G6PI        0.000000e+00   1000.0
GAPD        0.000000e+00   1000.0
GAPDi_nadp  0.000000e+00   1000.0
GLYOX_1    -1.857455e-14   1000.0
HACD1      -1.000000e+01   1000.0
HACD1i      0.000000e+00   1000.0
HACD8      -5.016302e-14   1000.0
HACD8i      0.000000e+00   1000.0
HSDxi       0.000000e+00   1000.0
HSDy        0.000000e+00   1000.0
LGTHL       0.000000e+00   1000.0
NADTRHD     0.000000e+00   1000.0
P5CR        0.000000e+

In [7]:
# Create a comprehensive list of the exact reactions to deactivate (0, 0)
internal_duplicates_to_block = internal_loops.index.tolist()

# Apply the block to your model
blocked_count = 0
for rxn_id in internal_duplicates_to_block:
    if rxn_id in model.reactions:
        model.reactions.get_by_id(rxn_id).bounds = (0, 0)
        blocked_count += 1

print(f"Successfully deactivated {blocked_count} internal looping reactions.")

Successfully deactivated 27 internal looping reactions.


In [8]:
# remote unused mets
cobra.manipulation.prune_unused_metabolites(model)

(<Model model1 at 0x7f88dcaaeb30>, [])

In [10]:
cobra.io.write_sbml_model(model, "../models/model_4.xml")